In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("ANTHROPIC_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"ANTHROPIC_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("ANTHROPIC_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'ANTHROPIC_API_KEY', and load_dotenv() ran without error.")

ANTHROPIC_API_KEY loaded (108 characters): sk-a...4wAA


In [3]:
%pip install anthropic


   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 6.2 MB/s  0:00:00

   ---------------------------------------- 0/2 [docstring-parser]
   -------------------- ------------------- 1/2 [anthropic]
   -------------------- ------------------- 1/2 [anthropic]
   -------------------- ------------------- 1/2 [anthropic]
   -------------------- ------------------- 1/2 [anthropic]
   -------------------- ------------------- 1/2 [anthropic]
   -------------------- ------------------- 1/2 [anthropic]
   -------------------- ------------------- 1/2 [anthropic]
   -------------------- ------------------- 1/2 [anthropic]
   -------------------- ------------------- 1/2 [anthropic]
   -------------------- ------------------- 1/2 [anthropic]
   -------------------- ------------------- 1/2 [anthropic]
   -------------------- ------------------- 1/2 [anthropic]



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
"""
Phase 2 triple extraction: full corpus, claude-sonnet-5 only.

Same pipeline as the Luna script (run_phase2_luna_full.py), with the model
swapped to Sonnet 5 for a real, reproducible accuracy/cost comparison run
via the API -- rather than an ad hoc chat test, which is not equivalent
(no fixed system/user separation, no temperature control, conversational
context bleed between chunks). See discussion in-thread for why a chat
comparison isn't a substitute for this.

Note: this required swapping the API client from openai to anthropic, and
the call syntax from client.chat.completions.create(...) to
client.messages.create(...), since Claude models are not served through
the OpenAI-compatible endpoint. Everything else -- prompt loading, chunk
corpus loading, checkpointing, retry logic, output flattening, Excel
sanitization -- is unchanged from the Luna script.

This is a long run (1000+ chunks), so:
    - checkpoints progress to disk every CHECKPOINT_EVERY chunks, so a crash
      or interruption does not lose completed work
    - resumes automatically from the last checkpoint on re-run
    - retries transient connection/timeout errors with backoff before giving
      up on a chunk
    - writes the flattened triples table incrementally, not just at the end

Produces:
    - model_comparison_output/phase2_sonnet_full_raw.xlsx      one row per chunk
    - model_comparison_output/phase2_sonnet_full_triples.xlsx  one row per triple
    - model_comparison_output/phase2_sonnet_full_checkpoint.jsonl  resume state

Requires a .env file with:
    ANTHROPIC_API_KEY=...

Usage:
    python run_phase2_sonnet_full.py
"""

import os
import re
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise RuntimeError("ANTHROPIC_API_KEY not found. Add it to your .env file.")

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

CHUNKS_PATH = r"C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\PDF_PREPROCESSING_INTO_CHUNKS\chunks_all.parquet"
PROMPT_PATH = "phase2_extraction_prompt.md"
OUTPUT_DIR = Path("model_comparison_output")
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_NAME = "claude-sonnet-5"
# $/1M tokens, (input, output) -- confirmed from official docs.claude.com
# pricing table for claude-sonnet-5: $2/MTok input, $10/MTok output.
MODEL_PRICING = {"claude-sonnet-5": (2.00, 10.00)}

MAX_OUTPUT_TOKENS = 32768  # Anthropic requires an explicit max_tokens per call.
# claude-sonnet-5 supports up to 128K output tokens, so this is not the
# model's ceiling -- it's a generous but bounded cap. Raised from 16384
# after the first run truncated dense results_discussion sections; set
# high enough that no realistic chunk should hit it, while still capping
# worst-case per-call cost if something ever generates runaway output.

CHECKPOINT_PATH = OUTPUT_DIR / "phase2_sonnet_full_checkpoint.jsonl"
RAW_OUTPUT_PATH = OUTPUT_DIR / "phase2_sonnet_full_raw.xlsx"
TRIPLES_OUTPUT_PATH = OUTPUT_DIR / "phase2_sonnet_full_triples.xlsx"

CHECKPOINT_EVERY = 25     # flush checkpoint to disk every N chunks

# Connection-type errors (wifi drop, DNS blip, timeout) are treated as
# recoverable and retried patiently, since the run is long and unattended.
# Non-connection errors (e.g. a genuine 400 on malformed input) are not
# worth retrying the same way, they fail fast after a couple of tries.
CONNECTION_MAX_RETRIES = 20      # up to 20 retries for connection-type errors
CONNECTION_RETRY_BACKOFF_SEC = 15  # 15,30,60,120,240,300(capped)...
CONNECTION_RETRY_CAP_SEC = 300     # never wait longer than 5 min between retries
OTHER_MAX_RETRIES = 3
OTHER_RETRY_BACKOFF_SEC = 10


# ---------------------------------------------------------------------------
# Fail fast on a bad key
# ---------------------------------------------------------------------------

def verify_anthropic_key():
    from anthropic import Anthropic
    client = Anthropic(api_key=ANTHROPIC_API_KEY)
    try:
        # Cheapest possible real call to confirm the key/model work before
        # spending a full corpus run on a bad key, same intent as the
        # Luna script's client.models.list() check.
        client.messages.create(
            model=MODEL_NAME,
            max_tokens=1,
            messages=[{"role": "user", "content": "hi"}],
        )
    except Exception as e:
        raise RuntimeError(
            f"Anthropic API key/model check failed before running any chunks: {e}\n"
            "Fix ANTHROPIC_API_KEY in .env, confirm MODEL_NAME is correct, and re-run."
        )
    print("Anthropic API key verified.")


# ---------------------------------------------------------------------------
# Load prompt + full corpus
# ---------------------------------------------------------------------------

def load_system_prompt(path: str) -> str:
    text = Path(path).read_text(encoding="utf-8")
    text = re.sub(r"^#\s+Phase 2.*\n", "", text, count=1)
    return text.strip()


def load_full_corpus() -> pd.DataFrame:
    df = pd.read_parquet(CHUNKS_PATH)
    before = len(df)
    df = df[df["chunk_text"].str.strip().str.len() > 200].reset_index(drop=True)
    print(f"Loaded {before} chunks, {len(df)} after length filter (>200 chars).")
    return df


SYSTEM_PROMPT = load_system_prompt(PROMPT_PATH)
corpus_df = load_full_corpus()


def build_user_message(row: pd.Series) -> str:
    return (
        f"pmid: {row['doi']}\n"
        f"section: {row['section']}\n\n"
        f"chunk_text:\n{row['chunk_text']}"
    )


# ---------------------------------------------------------------------------
# JSON parsing helper
# ---------------------------------------------------------------------------

def parse_triples(raw_text: str):
    """Robust JSON extraction, not just fence-stripping.

    Sonnet 5's text block can be messier than a clean fenced JSON array:
    observed real cases include a stray leading colon before the fence
    (":\`\`\`json..."), and the model's own reasoning prose leaking into the
    text block ahead of the actual JSON. A naive strip-fences-and-parse
    approach fails on both, silently losing real triples. This tries a
    clean parse first, then falls back to scanning for the first '[' ...
    matching ']' span in the text that parses as valid JSON, so triples
    aren't lost just because of surrounding junk.
    """
    text = raw_text.strip()
    text = re.sub(r"^[:\s]*```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```\s*$", "", text)
    text = text.strip()

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            parsed = [parsed]
        return parsed, None
    except json.JSONDecodeError:
        pass

    for match_start in [m.start() for m in re.finditer(r"\[", text)]:
        candidate = text[match_start:]
        last_close = candidate.rfind("]")
        if last_close == -1:
            continue
        candidate = candidate[:last_close + 1]
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                parsed = [parsed]
            return parsed, None
        except json.JSONDecodeError:
            continue

    # Genuine non-JSON prose response (e.g. the model explains in plain
    # English instead of returning [], or reasoning got cut off before any
    # JSON was written) -- not recoverable by parsing, worth surfacing
    # distinctly from a malformed-but-present JSON error.
    return None, f"JSON parse error: no valid JSON array found in response (starts with: {text[:80]!r})"


# ---------------------------------------------------------------------------
# Checkpointing: one JSON line per completed chunk result
# ---------------------------------------------------------------------------

def load_checkpoint() -> dict:
    """Returns {chunk_id: result_dict} for chunks completed successfully.

    Chunks that permanently failed (parse_error set after retries exhausted)
    are NOT treated as done here, so a fresh run automatically retries them
    again rather than leaving them lost forever. Successful chunks
    (parse_error is None) are the only ones skipped.
    """
    done = {}
    failed_count = 0
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                    if rec.get("parse_error") is None:
                        done[rec["chunk_id"]] = rec
                    else:
                        failed_count += 1
                except json.JSONDecodeError:
                    continue
        msg = f"Resuming: {len(done)} chunks completed successfully in checkpoint."
        if failed_count:
            msg += f" {failed_count} previously-failed chunk(s) will be retried."
        print(msg)
    return done


def load_all_checkpoint_records() -> dict:
    """Returns {chunk_id: latest_record} for every chunk ever attempted,
    successful or not. Used only for final reporting, so permanently-failed
    chunks remain visible in the output files rather than disappearing.
    If a chunk_id appears more than once (retried across separate runs),
    the most recent (last) record for that id wins.
    """
    records = {}
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                    records[rec["chunk_id"]] = rec
                except json.JSONDecodeError:
                    continue
    return records


def append_checkpoint(record: dict):
    with open(CHECKPOINT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")


# ---------------------------------------------------------------------------
# Sonnet caller with retry on transient errors
# ---------------------------------------------------------------------------

def call_sonnet(client, user_msg: str):
    """Anthropic's Messages API takes system prompt as a top-level `system`
    parameter (not a message with role="system"), and requires an explicit
    max_tokens. `temperature` is omitted entirely: Claude Sonnet 5 has
    deprecated the parameter (confirmed via the API's own 400 error,
    "`temperature` is deprecated..."), so the model always runs at its
    default behavior, the same situation as Luna, which also rejected any
    explicit temperature value, just for a different underlying reason.

    effort="low" is set explicitly. Per Anthropic's own docs (confirmed
    via docs.claude.com/build-with-claude/effort and /adaptive-thinking):
    "Claude Sonnet 5 defaults to high effort on the Claude API" when no
    effort is specified, and "At high and max effort levels, Claude may
    think more extensively and can be more likely to exhaust the
    max_tokens budget." That default-high-effort behavior is exactly what
    caused the earlier NO_TEXT_BLOCK and mid-JSON truncation failures: the
    model's reasoning (in the thinking/text blocks) consumed the token
    budget before finishing the JSON answer. This is a closed-schema
    extraction task, not multi-step agentic reasoning, so low effort is
    the right fit -- Anthropic's own guidance recommends low effort for
    "simple classification tasks... high-volume use cases where marginal
    quality improvements don't justify additional latency or spend,"
    which describes this batch extraction job well.

    Uses streaming rather than a single blocking call. With max_tokens set
    high enough to avoid truncation (32768), Anthropic requires streaming
    for any request that could plausibly run past 10 minutes and refuses
    non-streaming calls above that threshold with a 400 error. Streaming
    avoids that limit entirely. Content blocks are read directly from the
    final message rather than via get_final_text(), since that convenience
    method throws outright if no text block is present at all (which can
    happen if a response is 100% thinking content), and thinking blocks
    must be filtered out from the collected text either way.
    """
    with client.messages.stream(
        model=MODEL_NAME,
        max_tokens=MAX_OUTPUT_TOKENS,
        system=SYSTEM_PROMPT,
        output_config={"effort": "low"},
        messages=[
            {"role": "user", "content": user_msg},
        ],
    ) as stream:
        stream.until_done()
        final_message = stream.get_final_message()

    # Read content blocks directly rather than via get_final_text(), which
    # throws if there is no text block at all. Sonnet 5 supports adaptive
    # thinking, so a response can legitimately contain a "thinking" block
    # (the model's reasoning, not the answer) alongside or instead of a
    # "text" block. We want only the text block(s); if none exist, that's
    # a real failure worth surfacing clearly rather than crashing on a
    # library convenience method.
    block_types = [block.type for block in final_message.content]
    text_blocks = [block.text for block in final_message.content if block.type == "text"]

    if not text_blocks:
        raise RuntimeError(
            f"NO_TEXT_BLOCK: response contained no text content block "
            f"(block types returned: {block_types}). This usually means "
            f"the model only produced reasoning/thinking content and never "
            f"reached a text answer, often because max_tokens was used up "
            f"by thinking tokens before any output could be written."
        )

    raw_output = "".join(text_blocks)

    usage = final_message.usage
    input_tokens = usage.input_tokens if usage else None
    output_tokens = usage.output_tokens if usage else None

    # Anthropic reports why generation stopped. If it hit the max_tokens
    # ceiling, the JSON is almost certainly truncated and will fail to
    # parse -- surface that clearly rather than letting it show up only as
    # an opaque JSON parse error later.
    if final_message.stop_reason == "max_tokens":
        raise RuntimeError(
            f"TRUNCATED_OUTPUT: response hit max_tokens ({MAX_OUTPUT_TOKENS}) "
            f"before finishing; output_tokens={output_tokens}, "
            f"block types returned: {block_types}. Raise MAX_OUTPUT_TOKENS "
            f"further if this recurs, or disable/limit thinking if thinking "
            f"tokens are consuming the budget before any text is written."
        )

    return raw_output, input_tokens, output_tokens


def run_chunk_with_retry(client, row: pd.Series) -> dict:
    user_msg = build_user_message(row)
    attempt = 0
    while True:
        attempt += 1
        start = time.time()
        try:
            raw_output, input_tokens, output_tokens = call_sonnet(client, user_msg)
            elapsed = time.time() - start
            triples, err = parse_triples(raw_output)
            return {
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": raw_output,
                "n_triples": len(triples) if triples is not None else None,
                "parse_error": err,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "latency_sec": round(elapsed, 2),
            }
        except Exception as e:
            err_str = str(e)
            is_connection = any(s in err_str for s in
                                 ["Connection", "connection", "timeout", "Timeout",
                                  "503", "502", "504", "429", "APIConnectionError",
                                  "RemoteProtocolError", "ReadTimeout", "overloaded_error",
                                  "NO_TEXT_BLOCK"])
            # NO_TEXT_BLOCK is included here even though it's raised by our
            # own code, not the API client. Observed in testing: an empty
            # response (zero content blocks, ~0 tokens billed, ~3s latency)
            # on a chunk that succeeded normally both before and after,
            # with unremarkable source text -- the signature of a transient
            # API-side hiccup rather than a genuine content problem. Worth
            # the patient connection-style retry rather than failing fast.

            if is_connection:
                max_retries = CONNECTION_MAX_RETRIES
                wait = min(
                    CONNECTION_RETRY_BACKOFF_SEC * (2 ** (attempt - 1)),
                    CONNECTION_RETRY_CAP_SEC,
                )
            else:
                max_retries = OTHER_MAX_RETRIES
                wait = OTHER_RETRY_BACKOFF_SEC * (2 ** (attempt - 1))

            if attempt <= max_retries:
                kind = "connection" if is_connection else "other"
                print(f"    retry {attempt}/{max_retries} ({kind} error) for "
                      f"{row['id']} in {wait}s ({err_str[:120]})")
                time.sleep(wait)
                continue

            return {
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": None,
                "n_triples": None,
                "parse_error": f"API error after {attempt} attempt(s): {e}",
                "input_tokens": None,
                "output_tokens": None,
                "latency_sec": None,
            }


# ---------------------------------------------------------------------------
# Cost estimate
# ---------------------------------------------------------------------------

def add_cost_column(df: pd.DataFrame) -> pd.DataFrame:
    in_price, out_price = MODEL_PRICING[MODEL_NAME]
    def _cost(r):
        if pd.isna(r["input_tokens"]) or pd.isna(r["output_tokens"]):
            return None
        return (r["input_tokens"] / 1_000_000 * in_price) + (r["output_tokens"] / 1_000_000 * out_price)
    df["est_cost_usd"] = df.apply(_cost, axis=1)
    return df


# ---------------------------------------------------------------------------
# Flatten raw_output JSON into one row per triple
# ---------------------------------------------------------------------------

TRIPLE_FIELDS = [
    "pmid", "source", "source_type", "material", "interaction", "target",
    "target_type", "compared_property", "reported_value", "claim_status",
    "flagged_phrase", "corresponding_sentence",
]


def flatten_triples(raw_df: pd.DataFrame) -> pd.DataFrame:
    flat_rows = []
    for _, row in raw_df.iterrows():
        if pd.isna(row["raw_output"]):
            continue
        triples, err = parse_triples(row["raw_output"])
        if triples is None:
            continue
        for t in triples:
            flat_row = {
                "chunk_id": row["chunk_id"],
                "doi": row["doi"],
                "section": row["section"],
            }
            for field in TRIPLE_FIELDS:
                flat_row[field] = t.get(field, "")
            flat_rows.append(flat_row)
    return pd.DataFrame(flat_rows)


# ---------------------------------------------------------------------------
# Excel-safe string sanitization
# ---------------------------------------------------------------------------

# openpyxl (used by pandas.to_excel) rejects certain control characters that
# XML cannot encode. These can show up in raw_output (rare model artifacts)
# or, more commonly, inside parse_error messages that embed the entire raw
# response text when JSON parsing fails. Without sanitizing, a single bad
# cell can throw IllegalCharacterError and abort the whole file write, even
# after every chunk has already been successfully processed and checkpointed.
_ILLEGAL_XLSX_CHARS_RE = re.compile(
    r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]"  # control chars except \t \n \r
)


def sanitize_for_excel(value):
    if isinstance(value, str):
        return _ILLEGAL_XLSX_CHARS_RE.sub("", value)
    return value


def sanitize_dataframe_for_excel(df: pd.DataFrame) -> pd.DataFrame:
    try:
        return df.map(sanitize_for_excel)          # pandas >= 2.1
    except AttributeError:
        return df.applymap(sanitize_for_excel)      # pandas < 2.1


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    from anthropic import Anthropic

    verify_anthropic_key()
    client = Anthropic(api_key=ANTHROPIC_API_KEY)

    done = load_checkpoint()
    remaining = corpus_df[~corpus_df["id"].isin(done.keys())].reset_index(drop=True)
    print(f"{len(remaining)} chunks remaining out of {len(corpus_df)} total.\n")

    # Running cost tracking, for visibility only (no budget cap). Starts
    # from whatever was already spent in a prior partial run (read back
    # from the checkpoint), so resuming after an interruption doesn't lose
    # track of real cumulative spend.
    in_price, out_price = MODEL_PRICING[MODEL_NAME]
    running_cost = 0.0
    for rec in done.values():
        if rec.get("input_tokens") is not None and rec.get("output_tokens") is not None:
            running_cost += (rec["input_tokens"] / 1_000_000 * in_price) + \
                             (rec["output_tokens"] / 1_000_000 * out_price)

    since_last_flush = 0
    start_time = time.time()

    for i, row in remaining.iterrows():
        result = run_chunk_with_retry(client, row)
        append_checkpoint(result)
        since_last_flush += 1

        if result.get("input_tokens") is not None and result.get("output_tokens") is not None:
            running_cost += (result["input_tokens"] / 1_000_000 * in_price) + \
                             (result["output_tokens"] / 1_000_000 * out_price)

        status = "ok" if result["parse_error"] is None else "FAILED"
        n_tri = result["n_triples"] if result["n_triples"] is not None else "-"
        lat = f"{result['latency_sec']:.1f}s" if result["latency_sec"] else "-"
        overall_done = len(done) + i + 1
        print(f"  [{overall_done}/{len(corpus_df)}] {status} "
              f"({n_tri} triples, {lat}) {row['id']}  [running cost: ${running_cost:.2f}]")

        if since_last_flush >= CHECKPOINT_EVERY:
            elapsed_min = (time.time() - start_time) / 60
            print(f"  --- checkpoint: {overall_done}/{len(corpus_df)} done, "
                  f"{elapsed_min:.1f} min elapsed, ${running_cost:.2f} spent so far ---")
            since_last_flush = 0

    print("\nAll chunks processed. Building final output files...")

    # Reload everything from checkpoint (source of truth) rather than
    # relying on in-memory state, in case this run resumed a prior one.
    # Uses load_all_checkpoint_records so permanently-failed chunks still
    # show up in the raw output file for visibility, not just successes.
    all_records = load_all_checkpoint_records()
    combined = pd.DataFrame(list(all_records.values()))
    combined["model"] = MODEL_NAME
    combined = add_cost_column(combined)
    combined = sanitize_dataframe_for_excel(combined)

    try:
        combined.to_excel(RAW_OUTPUT_PATH, index=False)
        print(f"Raw results ({len(combined)} chunks) -> {RAW_OUTPUT_PATH}")
    except Exception as e:
        # Never lose a completed run over a write-format issue. Fall back to
        # CSV, which has no character restrictions, so the data survives
        # even if something Excel-specific still trips up openpyxl.
        csv_fallback = RAW_OUTPUT_PATH.with_suffix(".csv")
        combined.to_csv(csv_fallback, index=False, encoding="utf-8")
        print(f"WARNING: Excel write failed ({e}). "
              f"Saved as CSV instead -> {csv_fallback}")

    triples_df = flatten_triples(combined)
    triples_df = sanitize_dataframe_for_excel(triples_df)
    try:
        triples_df.to_excel(TRIPLES_OUTPUT_PATH, index=False)
        print(f"Flattened triples ({len(triples_df)} rows) -> {TRIPLES_OUTPUT_PATH}")
    except Exception as e:
        csv_fallback = TRIPLES_OUTPUT_PATH.with_suffix(".csv")
        triples_df.to_csv(csv_fallback, index=False, encoding="utf-8")
        print(f"WARNING: Excel write failed ({e}). "
              f"Saved as CSV instead -> {csv_fallback}")

    n_failed = combined["parse_error"].notna().sum()
    total_cost = combined["est_cost_usd"].sum()
    total_latency_hr = combined["latency_sec"].sum() / 3600 if combined["latency_sec"].notna().any() else 0
    print(f"\nChunks run: {len(combined)}  |  Failed: {n_failed}  |  "
          f"Est. total cost: ${total_cost:.2f}  |  "
          f"Total model time: {total_latency_hr:.1f} hr")
    if n_failed:
        print(f"\n{n_failed} chunks failed after retries. Re-running this script "
              f"will retry only the failed/missing chunks (resume is automatic).")


if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == "--rebuild-only":
        # Regenerate output files from the existing checkpoint without
        # calling the API again. Use this if extraction already finished
        # (or got far enough) but the final Excel write failed.
        print("Rebuilding output files from checkpoint only, no API calls...")
        all_records = load_all_checkpoint_records()
        combined = pd.DataFrame(list(all_records.values()))
        combined["model"] = MODEL_NAME
        combined = add_cost_column(combined)
        combined = sanitize_dataframe_for_excel(combined)

        try:
            combined.to_excel(RAW_OUTPUT_PATH, index=False)
            print(f"Raw results ({len(combined)} chunks) -> {RAW_OUTPUT_PATH}")
        except Exception as e:
            csv_fallback = RAW_OUTPUT_PATH.with_suffix(".csv")
            combined.to_csv(csv_fallback, index=False, encoding="utf-8")
            print(f"WARNING: Excel write failed ({e}). Saved as CSV -> {csv_fallback}")

        triples_df = flatten_triples(combined)
        triples_df = sanitize_dataframe_for_excel(triples_df)
        try:
            triples_df.to_excel(TRIPLES_OUTPUT_PATH, index=False)
            print(f"Flattened triples ({len(triples_df)} rows) -> {TRIPLES_OUTPUT_PATH}")
        except Exception as e:
            csv_fallback = TRIPLES_OUTPUT_PATH.with_suffix(".csv")
            triples_df.to_csv(csv_fallback, index=False, encoding="utf-8")
            print(f"WARNING: Excel write failed ({e}). Saved as CSV -> {csv_fallback}")

        n_failed = combined["parse_error"].notna().sum()
        print(f"\nChunks in checkpoint: {len(combined)}  |  Failed: {n_failed}")
    else:
        main()

<>:150: SyntaxWarning: invalid escape sequence '\`'
<>:150: SyntaxWarning: invalid escape sequence '\`'
C:\Users\olagunju\AppData\Local\Temp\ipykernel_10412\3284758347.py:150: SyntaxWarning: invalid escape sequence '\`'
  """Robust JSON extraction, not just fence-stripping.


Loaded 1108 chunks, 1104 after length filter (>200 chars).
Anthropic API key verified.
1104 chunks remaining out of 1104 total.

  [1/1104] ok (21 triples, 27.2s) 10-1002_cche-10383_abstract1  [running cost: $0.06]
  [2/1104] ok (0 triples, 3.5s) 10-1002_cche-10383_intro1  [running cost: $0.08]
  [3/1104] ok (0 triples, 3.1s) 10-1002_cche-10383_methods1  [running cost: $0.11]
  [4/1104] ok (14 triples, 22.7s) 10-1002_cche-10383_results_discussion1  [running cost: $0.17]
  [5/1104] ok (16 triples, 33.3s) 10-1002_cche-10383_results_discussion2  [running cost: $0.24]
  [6/1104] ok (9 triples, 14.7s) 10-1002_cche-10589_abstract1  [running cost: $0.28]
  [7/1104] ok (0 triples, 4.2s) 10-1002_cche-10589_intro1  [running cost: $0.30]
  [8/1104] ok (0 triples, 2.7s) 10-1002_cche-10589_methods1  [running cost: $0.33]
  [9/1104] ok (0 triples, 2.9s) 10-1002_cche-10589_methods2  [running cost: $0.35]
  [10/1104] ok (15 triples, 25.3s) 10-1002_cche-10589_results_discussion1  [running cost: $0.41]


C:\Users\olagunju\AppData\Local\Temp\ipykernel_10412\3284758347.py:150: SyntaxWarning: invalid escape sequence '\`'
  """Robust JSON extraction, not just fence-stripping.


AttributeError: 'int' object has no attribute 'get'

# rebuild output file

In [4]:
"""
Phase 2 triple extraction: full corpus, claude-sonnet-5 only.

Same pipeline as the Luna script (run_phase2_luna_full.py), with the model
swapped to Sonnet 5 for a real, reproducible accuracy/cost comparison run
via the API -- rather than an ad hoc chat test, which is not equivalent
(no fixed system/user separation, no temperature control, conversational
context bleed between chunks). See discussion in-thread for why a chat
comparison isn't a substitute for this.

Note: this required swapping the API client from openai to anthropic, and
the call syntax from client.chat.completions.create(...) to
client.messages.create(...), since Claude models are not served through
the OpenAI-compatible endpoint. Everything else -- prompt loading, chunk
corpus loading, checkpointing, retry logic, output flattening, Excel
sanitization -- is unchanged from the Luna script.

This is a long run (1000+ chunks), so:
    - checkpoints progress to disk every CHECKPOINT_EVERY chunks, so a crash
      or interruption does not lose completed work
    - resumes automatically from the last checkpoint on re-run
    - retries transient connection/timeout errors with backoff before giving
      up on a chunk
    - writes the flattened triples table incrementally, not just at the end

Produces:
    - model_comparison_output/phase2_sonnet_full_raw.xlsx      one row per chunk
    - model_comparison_output/phase2_sonnet_full_triples.xlsx  one row per triple
    - model_comparison_output/phase2_sonnet_full_checkpoint.jsonl  resume state

Requires a .env file with:
    ANTHROPIC_API_KEY=...

Usage:
    python run_phase2_sonnet_full.py
"""

import os
import re
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise RuntimeError("ANTHROPIC_API_KEY not found. Add it to your .env file.")

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

CHUNKS_PATH = r"C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\PDF_PREPROCESSING_INTO_CHUNKS\chunks_all.parquet"
PROMPT_PATH = "phase2_extraction_prompt.md"
OUTPUT_DIR = Path("model_comparison_output")
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_NAME = "claude-sonnet-5"
# $/1M tokens, (input, output) -- confirmed from official docs.claude.com
# pricing table for claude-sonnet-5: $2/MTok input, $10/MTok output.
MODEL_PRICING = {"claude-sonnet-5": (2.00, 10.00)}

MAX_OUTPUT_TOKENS = 32768  # Anthropic requires an explicit max_tokens per call.
# claude-sonnet-5 supports up to 128K output tokens, so this is not the
# model's ceiling -- it's a generous but bounded cap. Raised from 16384
# after the first run truncated dense results_discussion sections; set
# high enough that no realistic chunk should hit it, while still capping
# worst-case per-call cost if something ever generates runaway output.

CHECKPOINT_PATH = OUTPUT_DIR / "phase2_sonnet_full_checkpoint.jsonl"
RAW_OUTPUT_PATH = OUTPUT_DIR / "phase2_sonnet_full_raw.xlsx"
TRIPLES_OUTPUT_PATH = OUTPUT_DIR / "phase2_sonnet_full_triples.xlsx"

CHECKPOINT_EVERY = 25     # flush checkpoint to disk every N chunks

# Connection-type errors (wifi drop, DNS blip, timeout) are treated as
# recoverable and retried patiently, since the run is long and unattended.
# Non-connection errors (e.g. a genuine 400 on malformed input) are not
# worth retrying the same way, they fail fast after a couple of tries.
CONNECTION_MAX_RETRIES = 20      # up to 20 retries for connection-type errors
CONNECTION_RETRY_BACKOFF_SEC = 15  # 15,30,60,120,240,300(capped)...
CONNECTION_RETRY_CAP_SEC = 300     # never wait longer than 5 min between retries
OTHER_MAX_RETRIES = 3
OTHER_RETRY_BACKOFF_SEC = 10

# Notebook-friendly run control: set this instead of using a command-line
# flag (sys.argv doesn't work meaningfully in a notebook cell). Options:
#   "full"           -- normal run: resume/continue extraction, then build
#                        output files at the end. This is what you want
#                        most of the time, including the very first run.
#   "rebuild_only"   -- skip the API entirely and just regenerate the two
#                        output Excel files from whatever is already in the
#                        checkpoint. Use this if extraction finished (or got
#                        far enough) but the final file-write step crashed
#                        (exactly what happened with the flatten_triples
#                        error) -- no cost, no re-running any chunks.
RUN_MODE = "full"


# ---------------------------------------------------------------------------
# Fail fast on a bad key
# ---------------------------------------------------------------------------

def verify_anthropic_key():
    from anthropic import Anthropic
    client = Anthropic(api_key=ANTHROPIC_API_KEY)
    try:
        # Cheapest possible real call to confirm the key/model work before
        # spending a full corpus run on a bad key, same intent as the
        # Luna script's client.models.list() check.
        client.messages.create(
            model=MODEL_NAME,
            max_tokens=1,
            messages=[{"role": "user", "content": "hi"}],
        )
    except Exception as e:
        raise RuntimeError(
            f"Anthropic API key/model check failed before running any chunks: {e}\n"
            "Fix ANTHROPIC_API_KEY in .env, confirm MODEL_NAME is correct, and re-run."
        )
    print("Anthropic API key verified.")


# ---------------------------------------------------------------------------
# Load prompt + full corpus
# ---------------------------------------------------------------------------

def load_system_prompt(path: str) -> str:
    text = Path(path).read_text(encoding="utf-8")
    text = re.sub(r"^#\s+Phase 2.*\n", "", text, count=1)
    return text.strip()


def load_full_corpus() -> pd.DataFrame:
    df = pd.read_parquet(CHUNKS_PATH)
    before = len(df)
    df = df[df["chunk_text"].str.strip().str.len() > 200].reset_index(drop=True)
    print(f"Loaded {before} chunks, {len(df)} after length filter (>200 chars).")
    return df


SYSTEM_PROMPT = load_system_prompt(PROMPT_PATH)
corpus_df = load_full_corpus()


def build_user_message(row: pd.Series) -> str:
    return (
        f"pmid: {row['doi']}\n"
        f"section: {row['section']}\n\n"
        f"chunk_text:\n{row['chunk_text']}"
    )


# ---------------------------------------------------------------------------
# JSON parsing helper
# ---------------------------------------------------------------------------

def parse_triples(raw_text: str):
    """Robust JSON extraction, not just fence-stripping.

    Sonnet 5's text block can be messier than a clean fenced JSON array:
    observed real cases include a stray leading colon before the fence
    (":\`\`\`json..."), and the model's own reasoning prose leaking into the
    text block ahead of the actual JSON. A naive strip-fences-and-parse
    approach fails on both, silently losing real triples. This tries a
    clean parse first, then falls back to scanning for the first '[' ...
    matching ']' span in the text that parses as valid JSON, so triples
    aren't lost just because of surrounding junk.
    """
    text = raw_text.strip()
    text = re.sub(r"^[:\s]*```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```\s*$", "", text)
    text = text.strip()

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            parsed = [parsed]
        return parsed, None
    except json.JSONDecodeError:
        pass

    for match_start in [m.start() for m in re.finditer(r"\[", text)]:
        candidate = text[match_start:]
        last_close = candidate.rfind("]")
        if last_close == -1:
            continue
        candidate = candidate[:last_close + 1]
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                parsed = [parsed]
            return parsed, None
        except json.JSONDecodeError:
            continue

    # Genuine non-JSON prose response (e.g. the model explains in plain
    # English instead of returning [], or reasoning got cut off before any
    # JSON was written) -- not recoverable by parsing, worth surfacing
    # distinctly from a malformed-but-present JSON error.
    return None, f"JSON parse error: no valid JSON array found in response (starts with: {text[:80]!r})"


# ---------------------------------------------------------------------------
# Checkpointing: one JSON line per completed chunk result
# ---------------------------------------------------------------------------

def load_checkpoint() -> dict:
    """Returns {chunk_id: result_dict} for chunks completed successfully.

    Chunks that permanently failed (parse_error set after retries exhausted)
    are NOT treated as done here, so a fresh run automatically retries them
    again rather than leaving them lost forever. Successful chunks
    (parse_error is None) are the only ones skipped.
    """
    done = {}
    failed_count = 0
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                    if rec.get("parse_error") is None:
                        done[rec["chunk_id"]] = rec
                    else:
                        failed_count += 1
                except json.JSONDecodeError:
                    continue
        msg = f"Resuming: {len(done)} chunks completed successfully in checkpoint."
        if failed_count:
            msg += f" {failed_count} previously-failed chunk(s) will be retried."
        print(msg)
    return done


def load_all_checkpoint_records() -> dict:
    """Returns {chunk_id: latest_record} for every chunk ever attempted,
    successful or not. Used only for final reporting, so permanently-failed
    chunks remain visible in the output files rather than disappearing.
    If a chunk_id appears more than once (retried across separate runs),
    the most recent (last) record for that id wins.
    """
    records = {}
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                    records[rec["chunk_id"]] = rec
                except json.JSONDecodeError:
                    continue
    return records


def append_checkpoint(record: dict):
    with open(CHECKPOINT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")


# ---------------------------------------------------------------------------
# Sonnet caller with retry on transient errors
# ---------------------------------------------------------------------------

def call_sonnet(client, user_msg: str):
    """Anthropic's Messages API takes system prompt as a top-level `system`
    parameter (not a message with role="system"), and requires an explicit
    max_tokens. `temperature` is omitted entirely: Claude Sonnet 5 has
    deprecated the parameter (confirmed via the API's own 400 error,
    "`temperature` is deprecated..."), so the model always runs at its
    default behavior, the same situation as Luna, which also rejected any
    explicit temperature value, just for a different underlying reason.

    effort="low" is set explicitly. Per Anthropic's own docs (confirmed
    via docs.claude.com/build-with-claude/effort and /adaptive-thinking):
    "Claude Sonnet 5 defaults to high effort on the Claude API" when no
    effort is specified, and "At high and max effort levels, Claude may
    think more extensively and can be more likely to exhaust the
    max_tokens budget." That default-high-effort behavior is exactly what
    caused the earlier NO_TEXT_BLOCK and mid-JSON truncation failures: the
    model's reasoning (in the thinking/text blocks) consumed the token
    budget before finishing the JSON answer. This is a closed-schema
    extraction task, not multi-step agentic reasoning, so low effort is
    the right fit -- Anthropic's own guidance recommends low effort for
    "simple classification tasks... high-volume use cases where marginal
    quality improvements don't justify additional latency or spend,"
    which describes this batch extraction job well.

    Uses streaming rather than a single blocking call. With max_tokens set
    high enough to avoid truncation (32768), Anthropic requires streaming
    for any request that could plausibly run past 10 minutes and refuses
    non-streaming calls above that threshold with a 400 error. Streaming
    avoids that limit entirely. Content blocks are read directly from the
    final message rather than via get_final_text(), since that convenience
    method throws outright if no text block is present at all (which can
    happen if a response is 100% thinking content), and thinking blocks
    must be filtered out from the collected text either way.
    """
    with client.messages.stream(
        model=MODEL_NAME,
        max_tokens=MAX_OUTPUT_TOKENS,
        system=SYSTEM_PROMPT,
        output_config={"effort": "low"},
        messages=[
            {"role": "user", "content": user_msg},
        ],
    ) as stream:
        stream.until_done()
        final_message = stream.get_final_message()

    # Read content blocks directly rather than via get_final_text(), which
    # throws if there is no text block at all. Sonnet 5 supports adaptive
    # thinking, so a response can legitimately contain a "thinking" block
    # (the model's reasoning, not the answer) alongside or instead of a
    # "text" block. We want only the text block(s); if none exist, that's
    # a real failure worth surfacing clearly rather than crashing on a
    # library convenience method.
    block_types = [block.type for block in final_message.content]
    text_blocks = [block.text for block in final_message.content if block.type == "text"]

    if not text_blocks:
        raise RuntimeError(
            f"NO_TEXT_BLOCK: response contained no text content block "
            f"(block types returned: {block_types}). This usually means "
            f"the model only produced reasoning/thinking content and never "
            f"reached a text answer, often because max_tokens was used up "
            f"by thinking tokens before any output could be written."
        )

    raw_output = "".join(text_blocks)

    usage = final_message.usage
    input_tokens = usage.input_tokens if usage else None
    output_tokens = usage.output_tokens if usage else None

    # Anthropic reports why generation stopped. If it hit the max_tokens
    # ceiling, the JSON is almost certainly truncated and will fail to
    # parse -- surface that clearly rather than letting it show up only as
    # an opaque JSON parse error later.
    if final_message.stop_reason == "max_tokens":
        raise RuntimeError(
            f"TRUNCATED_OUTPUT: response hit max_tokens ({MAX_OUTPUT_TOKENS}) "
            f"before finishing; output_tokens={output_tokens}, "
            f"block types returned: {block_types}. Raise MAX_OUTPUT_TOKENS "
            f"further if this recurs, or disable/limit thinking if thinking "
            f"tokens are consuming the budget before any text is written."
        )

    return raw_output, input_tokens, output_tokens


def run_chunk_with_retry(client, row: pd.Series) -> dict:
    user_msg = build_user_message(row)
    attempt = 0
    while True:
        attempt += 1
        start = time.time()
        try:
            raw_output, input_tokens, output_tokens = call_sonnet(client, user_msg)
            elapsed = time.time() - start
            triples, err = parse_triples(raw_output)
            return {
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": raw_output,
                "n_triples": len(triples) if triples is not None else None,
                "parse_error": err,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "latency_sec": round(elapsed, 2),
            }
        except Exception as e:
            err_str = str(e)
            is_connection = any(s in err_str for s in
                                 ["Connection", "connection", "timeout", "Timeout",
                                  "503", "502", "504", "429", "APIConnectionError",
                                  "RemoteProtocolError", "ReadTimeout", "overloaded_error",
                                  "NO_TEXT_BLOCK"])
            # NO_TEXT_BLOCK is included here even though it's raised by our
            # own code, not the API client. Observed in testing: an empty
            # response (zero content blocks, ~0 tokens billed, ~3s latency)
            # on a chunk that succeeded normally both before and after,
            # with unremarkable source text -- the signature of a transient
            # API-side hiccup rather than a genuine content problem. Worth
            # the patient connection-style retry rather than failing fast.

            if is_connection:
                max_retries = CONNECTION_MAX_RETRIES
                wait = min(
                    CONNECTION_RETRY_BACKOFF_SEC * (2 ** (attempt - 1)),
                    CONNECTION_RETRY_CAP_SEC,
                )
            else:
                max_retries = OTHER_MAX_RETRIES
                wait = OTHER_RETRY_BACKOFF_SEC * (2 ** (attempt - 1))

            if attempt <= max_retries:
                kind = "connection" if is_connection else "other"
                print(f"    retry {attempt}/{max_retries} ({kind} error) for "
                      f"{row['id']} in {wait}s ({err_str[:120]})")
                time.sleep(wait)
                continue

            return {
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": None,
                "n_triples": None,
                "parse_error": f"API error after {attempt} attempt(s): {e}",
                "input_tokens": None,
                "output_tokens": None,
                "latency_sec": None,
            }


# ---------------------------------------------------------------------------
# Cost estimate
# ---------------------------------------------------------------------------

def add_cost_column(df: pd.DataFrame) -> pd.DataFrame:
    in_price, out_price = MODEL_PRICING[MODEL_NAME]
    def _cost(r):
        if pd.isna(r["input_tokens"]) or pd.isna(r["output_tokens"]):
            return None
        return (r["input_tokens"] / 1_000_000 * in_price) + (r["output_tokens"] / 1_000_000 * out_price)
    df["est_cost_usd"] = df.apply(_cost, axis=1)
    return df


# ---------------------------------------------------------------------------
# Flatten raw_output JSON into one row per triple
# ---------------------------------------------------------------------------

TRIPLE_FIELDS = [
    "pmid", "source", "source_type", "material", "interaction", "target",
    "target_type", "compared_property", "reported_value", "claim_status",
    "flagged_phrase", "corresponding_sentence",
]


def flatten_triples(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Explode each chunk's parsed JSON list into one row per triple.

    Guards against malformed list contents: occasionally a parsed JSON
    array contains a non-dict element (e.g. a stray int/str that isn't a
    real triple object), which crashes a naive t.get(field, "") call.
    Skip anything that isn't a dict rather than crashing the whole
    flatten pass over one bad element in one chunk's output.
    """
    flat_rows = []
    skipped_non_dict = 0
    for _, row in raw_df.iterrows():
        if pd.isna(row["raw_output"]):
            continue
        triples, err = parse_triples(row["raw_output"])
        if triples is None:
            continue
        for t in triples:
            if not isinstance(t, dict):
                skipped_non_dict += 1
                continue
            flat_row = {
                "chunk_id": row["chunk_id"],
                "doi": row["doi"],
                "section": row["section"],
            }
            for field in TRIPLE_FIELDS:
                flat_row[field] = t.get(field, "")
            flat_rows.append(flat_row)
    if skipped_non_dict:
        print(f"  Note: skipped {skipped_non_dict} non-dict list element(s) "
              f"found inside otherwise-valid JSON arrays during flattening.")
    return pd.DataFrame(flat_rows)


# ---------------------------------------------------------------------------
# Excel-safe string sanitization
# ---------------------------------------------------------------------------

# openpyxl (used by pandas.to_excel) rejects certain control characters that
# XML cannot encode. These can show up in raw_output (rare model artifacts)
# or, more commonly, inside parse_error messages that embed the entire raw
# response text when JSON parsing fails. Without sanitizing, a single bad
# cell can throw IllegalCharacterError and abort the whole file write, even
# after every chunk has already been successfully processed and checkpointed.
_ILLEGAL_XLSX_CHARS_RE = re.compile(
    r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]"  # control chars except \t \n \r
)


def sanitize_for_excel(value):
    if isinstance(value, str):
        return _ILLEGAL_XLSX_CHARS_RE.sub("", value)
    return value


def sanitize_dataframe_for_excel(df: pd.DataFrame) -> pd.DataFrame:
    try:
        return df.map(sanitize_for_excel)          # pandas >= 2.1
    except AttributeError:
        return df.applymap(sanitize_for_excel)      # pandas < 2.1


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def build_output_files_from_checkpoint():
    """Reload everything from the checkpoint (source of truth) and write
    both output Excel files. Used both at the end of a normal run and by
    RUN_MODE = "rebuild_only" to regenerate outputs without calling the
    API again -- e.g. after the flatten_triples crash, where extraction
    had already finished and only the final file-write step failed.

    Uses load_all_checkpoint_records so permanently-failed chunks still
    show up in the raw output file for visibility, not just successes.
    """
    all_records = load_all_checkpoint_records()
    combined = pd.DataFrame(list(all_records.values()))
    combined["model"] = MODEL_NAME
    combined = add_cost_column(combined)
    combined = sanitize_dataframe_for_excel(combined)

    try:
        combined.to_excel(RAW_OUTPUT_PATH, index=False)
        print(f"Raw results ({len(combined)} chunks) -> {RAW_OUTPUT_PATH}")
    except Exception as e:
        # Never lose a completed run over a write-format issue. Fall back to
        # CSV, which has no character restrictions, so the data survives
        # even if something Excel-specific still trips up openpyxl.
        csv_fallback = RAW_OUTPUT_PATH.with_suffix(".csv")
        combined.to_csv(csv_fallback, index=False, encoding="utf-8")
        print(f"WARNING: Excel write failed ({e}). "
              f"Saved as CSV instead -> {csv_fallback}")

    triples_df = flatten_triples(combined)
    triples_df = sanitize_dataframe_for_excel(triples_df)
    try:
        triples_df.to_excel(TRIPLES_OUTPUT_PATH, index=False)
        print(f"Flattened triples ({len(triples_df)} rows) -> {TRIPLES_OUTPUT_PATH}")
    except Exception as e:
        csv_fallback = TRIPLES_OUTPUT_PATH.with_suffix(".csv")
        triples_df.to_csv(csv_fallback, index=False, encoding="utf-8")
        print(f"WARNING: Excel write failed ({e}). "
              f"Saved as CSV instead -> {csv_fallback}")

    n_failed = combined["parse_error"].notna().sum()
    total_cost = combined["est_cost_usd"].sum()
    total_latency_hr = combined["latency_sec"].sum() / 3600 if combined["latency_sec"].notna().any() else 0
    print(f"\nChunks in output: {len(combined)}  |  Failed: {n_failed}  |  "
          f"Est. total cost: ${total_cost:.2f}  |  "
          f"Total model time: {total_latency_hr:.1f} hr")
    if n_failed:
        print(f"\n{n_failed} chunks failed after retries. Re-running this "
              f"cell (with RUN_MODE = \"full\") will retry only the "
              f"failed/missing chunks (resume is automatic).")


def main():
    from anthropic import Anthropic

    verify_anthropic_key()
    client = Anthropic(api_key=ANTHROPIC_API_KEY)

    done = load_checkpoint()
    remaining = corpus_df[~corpus_df["id"].isin(done.keys())].reset_index(drop=True)
    print(f"{len(remaining)} chunks remaining out of {len(corpus_df)} total.\n")

    # Running cost tracking, for visibility only (no budget cap). Starts
    # from whatever was already spent in a prior partial run (read back
    # from the checkpoint), so resuming after an interruption doesn't lose
    # track of real cumulative spend.
    in_price, out_price = MODEL_PRICING[MODEL_NAME]
    running_cost = 0.0
    for rec in done.values():
        if rec.get("input_tokens") is not None and rec.get("output_tokens") is not None:
            running_cost += (rec["input_tokens"] / 1_000_000 * in_price) + \
                             (rec["output_tokens"] / 1_000_000 * out_price)

    since_last_flush = 0
    start_time = time.time()

    for i, row in remaining.iterrows():
        result = run_chunk_with_retry(client, row)
        append_checkpoint(result)
        since_last_flush += 1

        if result.get("input_tokens") is not None and result.get("output_tokens") is not None:
            running_cost += (result["input_tokens"] / 1_000_000 * in_price) + \
                             (result["output_tokens"] / 1_000_000 * out_price)

        status = "ok" if result["parse_error"] is None else "FAILED"
        n_tri = result["n_triples"] if result["n_triples"] is not None else "-"
        lat = f"{result['latency_sec']:.1f}s" if result["latency_sec"] else "-"
        overall_done = len(done) + i + 1
        print(f"  [{overall_done}/{len(corpus_df)}] {status} "
              f"({n_tri} triples, {lat}) {row['id']}  [running cost: ${running_cost:.2f}]")

        if since_last_flush >= CHECKPOINT_EVERY:
            elapsed_min = (time.time() - start_time) / 60
            print(f"  --- checkpoint: {overall_done}/{len(corpus_df)} done, "
                  f"{elapsed_min:.1f} min elapsed, ${running_cost:.2f} spent so far ---")
            since_last_flush = 0

    print("\nAll chunks processed. Building final output files...")
    build_output_files_from_checkpoint()


if RUN_MODE == "rebuild_only":
    print("Rebuilding output files from checkpoint only, no API calls...")
    build_output_files_from_checkpoint()
elif RUN_MODE == "full":
    main()
else:
    raise ValueError(f'RUN_MODE must be "full" or "rebuild_only", got {RUN_MODE!r}')

<>:163: SyntaxWarning: invalid escape sequence '\`'
<>:163: SyntaxWarning: invalid escape sequence '\`'
C:\Users\olagunju\AppData\Local\Temp\ipykernel_10412\1596629096.py:163: SyntaxWarning: invalid escape sequence '\`'
  """Robust JSON extraction, not just fence-stripping.


Loaded 1108 chunks, 1104 after length filter (>200 chars).
Anthropic API key verified.
Resuming: 1080 chunks completed successfully in checkpoint. 24 previously-failed chunk(s) will be retried.
24 chunks remaining out of 1104 total.

  [1081/1104] FAILED (- triples, 8.8s) 10-1002_jsfa-13233_results_discussion1  [running cost: $41.39]
  [1082/1104] ok (14 triples, 21.1s) 10-1002_jsfa-70829_results_discussion1  [running cost: $41.44]
  [1083/1104] ok (12 triples, 21.1s) 10-1002_jsfa-70829_results_discussion2  [running cost: $41.50]
  [1084/1104] FAILED (- triples, 6.5s) 10-1002_leg3-70005_abstract2  [running cost: $41.53]
  [1085/1104] ok (5 triples, 11.6s) 10-1002_leg3-70032_abstract1  [running cost: $41.57]
  [1086/1104] ok (1 triples, 8.9s) 10-1007_s00217-026-05065-0_results_discussion2  [running cost: $41.60]
  [1087/1104] ok (31 triples, 49.7s) 10-1007_s11694-013-9168-x_results_discussion1  [running cost: $41.70]
  [1088/1104] ok (13 triples, 20.5s) 10-1007_s13197-014-1681-3_results

# small chunk test

In [2]:
"""
Small-scale test for the Sonnet 5 extraction fix: runs ONLY the chunks from
2 specific articles (the ones that were failing in the full run) to confirm
the NO_TEXT_BLOCK fix actually works before spending more on a 1104-chunk run.

This does NOT touch the main checkpoint file, so it won't interfere with
your full-corpus run's progress -- it writes to a separate test checkpoint
and output file.

Usage:
    python test_sonnet_small.py
"""

import os
import re
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise RuntimeError("ANTHROPIC_API_KEY not found. Add it to your .env file.")

CHUNKS_PATH = r"C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\PDF_PREPROCESSING_INTO_CHUNKS\chunks_all.parquet"
PROMPT_PATH = "phase2_extraction_prompt.md"
MODEL_NAME = "claude-sonnet-5"
MAX_OUTPUT_TOKENS = 32768

# The two articles from your failing run -- includes every chunk type
# (abstract, intro, methods, results_discussion, conclusion) so this test
# covers the same variety of section lengths that triggered failures.
TEST_DOIS = ["10-1002_cche-10383", "10-1002_cche-10589"]

OUTPUT_DIR = Path("model_comparison_output")
OUTPUT_DIR.mkdir(exist_ok=True)
TEST_OUTPUT_PATH = OUTPUT_DIR / "sonnet_small_test_results.xlsx"


def load_system_prompt(path: str) -> str:
    text = Path(path).read_text(encoding="utf-8")
    text = re.sub(r"^#\s+Phase 2.*\n", "", text, count=1)
    return text.strip()


def build_user_message(row: pd.Series) -> str:
    return (
        f"pmid: {row['doi']}\n"
        f"section: {row['section']}\n\n"
        f"chunk_text:\n{row['chunk_text']}"
    )


def parse_triples(raw_text: str):
    """Robust JSON extraction -- see run_phase2_sonnet_full.py for the
    detailed rationale. Same logic here so this small test script gives an
    accurate read on what the full script will actually recover.
    """
    text = raw_text.strip()
    text = re.sub(r"^[:\s]*```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```\s*$", "", text)
    text = text.strip()

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            parsed = [parsed]
        return parsed, None
    except json.JSONDecodeError:
        pass

    for match_start in [m.start() for m in re.finditer(r"\[", text)]:
        candidate = text[match_start:]
        last_close = candidate.rfind("]")
        if last_close == -1:
            continue
        candidate = candidate[:last_close + 1]
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                parsed = [parsed]
            return parsed, None
        except json.JSONDecodeError:
            continue

    return None, f"JSON parse error: no valid JSON array found in response (starts with: {text[:80]!r})"


def call_sonnet(client, system_prompt: str, user_msg: str):
    with client.messages.stream(
        model=MODEL_NAME,
        max_tokens=MAX_OUTPUT_TOKENS,
        system=system_prompt,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": user_msg}],
    ) as stream:
        stream.until_done()
        final_message = stream.get_final_message()

    block_types = [block.type for block in final_message.content]
    text_blocks = [block.text for block in final_message.content if block.type == "text"]

    print(f"      block types returned: {block_types}")

    if not text_blocks:
        raise RuntimeError(f"NO_TEXT_BLOCK: only got {block_types}")

    raw_output = "".join(text_blocks)
    usage = final_message.usage
    input_tokens = usage.input_tokens if usage else None
    output_tokens = usage.output_tokens if usage else None

    if final_message.stop_reason == "max_tokens":
        raise RuntimeError(
            f"TRUNCATED_OUTPUT: hit max_tokens, block types: {block_types}"
        )

    return raw_output, input_tokens, output_tokens


def main():
    from anthropic import Anthropic

    client = Anthropic(api_key=ANTHROPIC_API_KEY)
    system_prompt = load_system_prompt(PROMPT_PATH)

    df = pd.read_parquet(CHUNKS_PATH)
    df = df[df["chunk_text"].str.strip().str.len() > 200].reset_index(drop=True)
    test_df = df[df["doi"].isin(TEST_DOIS)].reset_index(drop=True)

    print(f"Testing {len(test_df)} chunks from {TEST_DOIS}\n")

    results = []
    total_cost = 0.0
    in_price, out_price = 2.00, 10.00  # $/1M tokens

    for i, row in test_df.iterrows():
        print(f"  [{i+1}/{len(test_df)}] {row['id']}")
        start = time.time()
        try:
            raw_output, input_tokens, output_tokens = call_sonnet(
                client, system_prompt, build_user_message(row)
            )
            elapsed = time.time() - start
            triples, err = parse_triples(raw_output)
            cost = 0
            if input_tokens and output_tokens:
                cost = (input_tokens / 1e6 * in_price) + (output_tokens / 1e6 * out_price)
                total_cost += cost
            n_tri = len(triples) if triples is not None else None
            status = "ok" if err is None else f"PARSE_FAIL: {err}"
            print(f"      {status} ({n_tri} triples, {elapsed:.1f}s, ${cost:.3f}) "
                  f"[running total: ${total_cost:.3f}]")
            results.append({
                "chunk_id": row["id"], "doi": row["doi"], "section": row["section"],
                "status": status, "n_triples": n_tri, "raw_output": raw_output,
                "latency_sec": round(elapsed, 2), "cost_usd": round(cost, 4),
            })
        except Exception as e:
            elapsed = time.time() - start
            print(f"      FAILED: {e}")
            results.append({
                "chunk_id": row["id"], "doi": row["doi"], "section": row["section"],
                "status": f"FAILED: {e}", "n_triples": None, "raw_output": None,
                "latency_sec": round(elapsed, 2), "cost_usd": 0,
            })

    results_df = pd.DataFrame(results)
    results_df.to_excel(TEST_OUTPUT_PATH, index=False)
    print(f"\nTest complete. Total cost: ${total_cost:.3f}")
    print(f"Results -> {TEST_OUTPUT_PATH}")
    print(f"\n{results_df['status'].value_counts().to_string()}")


if __name__ == "__main__":
    main()

Testing 13 chunks from ['10-1002_cche-10383', '10-1002_cche-10589']

  [1/13] 10-1002_cche-10383_abstract1
      block types returned: ['thinking', 'text']
      ok (18 triples, 23.9s, $0.057) [running total: $0.057]
  [2/13] 10-1002_cche-10383_intro1
      block types returned: ['text']
      ok (0 triples, 2.7s, $0.023) [running total: $0.080]
  [3/13] 10-1002_cche-10383_methods1
      block types returned: ['text']
      ok (0 triples, 2.8s, $0.025) [running total: $0.105]
  [4/13] 10-1002_cche-10383_results_discussion1
      block types returned: ['thinking', 'text']
      ok (12 triples, 20.0s, $0.053) [running total: $0.158]
  [5/13] 10-1002_cche-10383_results_discussion2
      block types returned: ['thinking', 'text']
      ok (18 triples, 32.5s, $0.072) [running total: $0.230]
  [6/13] 10-1002_cche-10589_abstract1
      block types returned: ['thinking', 'text']
      ok (8 triples, 14.0s, $0.037) [running total: $0.267]
  [7/13] 10-1002_cche-10589_intro1
      block types ret